# Lektion 09 - Metakognition Designmuster


## Einrichtung

Dieses Notebook demonstriert das Metakognition-Entwurfsmuster unter Verwendung des Microsoft Agent Frameworks.

**Voraussetzungen:**
- Azure OpenAI-Bereitstellung, die über Umgebungsvariablen konfiguriert ist
- Azure CLI angemeldet (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Was ist Metakognition?

Metakognition ist **Nachdenken über das Denken**. Im Kontext von KI-Agenten bedeutet das, Agenten zu entwickeln, die:

- **Selbstreflexion** über ihre eigenen Ergebnisse und den Denkprozess
- **Fehler erkennen** und sich anständig erholen, anstatt stillschweigend zu scheitern
- **Bewerten**, ob ihre Antworten vollständig und hilfreich sind
- **Anpassen** ihrer Strategie, wenn ein erster Ansatz nicht funktioniert (z. B. Rückgriff auf ein Backup-System)

Ein metakognitiver Agent beantwortet nicht nur Fragen — er überwacht seine eigene Leistung und passt sich dabei in Echtzeit an.


## Primäre und Backup-Werkzeuge

Ein häufiges Metakognitionsmuster ist die **Fallback-Strategie**. Der Agent versucht zuerst ein primäres Werkzeug; wenn dieses fehlschlägt (z. B. ein 404-Fehler), erkennt der Agent den Fehler und wechselt transparent zu einem Backup-Werkzeug.

Dies spiegelt reale Systeme wider, bei denen primäre Dienste möglicherweise nicht verfügbar sind und Agenten das Problem selbst diagnostizieren müssen, bevor sie einen alternativen Weg wählen.

Unten definieren wir zwei Flugabfrage-Werkzeuge:
- **Primär** — umfasst Paris, Tokio und Barcelona
- **Backup** — umfasst Berlin, Sydney und New York City


In [ ]:
@tool(approval_mode="never_require")
def get_flight_times(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times for a destination (primary source)."""
    flights = {
        "Paris": "Departures: 08:00, 12:30, 17:45 — from $350",
        "Tokyo": "Departures: 11:00, 23:30 — from $890",
        "Barcelona": "Departures: 07:15, 14:00, 19:30 — from $280",
    }
    if destination in flights:
        return flights[destination]
    raise Exception(f"404: No flights found for {destination} in primary system")


@tool(approval_mode="never_require")
def get_flight_times_backup(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times from backup system (used when primary fails)."""
    backup_flights = {
        "Berlin": "Departures: 09:00, 16:00 — from $220",
        "Sydney": "Departures: 22:00 — from $1200",
        "New York City": "Departures: 06:00, 10:30, 15:00, 20:00 — from $450",
    }
    return backup_flights.get(
        destination,
        f"No flights found for {destination} in any system. Please try again later.",
    )

## Selbstreflektierender Agent mit Fehlerbehebung

Der unten stehende Agent wird angewiesen, zunächst das primäre Flugsystem zu testen, Fehler zu erkennen und transparent auf das Backupsystem zurückzugreifen. Nach jeder Antwort bewertet er kurz, ob er die Frage des Benutzers vollständig beantwortet hat.


In [ ]:
agent = client.as_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="FlightBookingAgent",
    instructions="""You are a flight booking agent with self-reflection capabilities.

When looking up flights:
1. Try the primary flight system first (get_flight_times)
2. If the primary system fails (404 error), acknowledge the error and try the backup system (get_flight_times_backup)
3. Always explain to the user what happened — be transparent about fallbacks
4. If both systems fail, apologize and suggest alternatives

After each response, briefly evaluate whether your answer was complete and helpful.""",
)

# Test with a destination in primary system
print("=== Test 1: Destination in primary system ===")
response = await agent.run(
    "What flights are available to Paris?",
    )
print(response)

# Test with a destination only in backup system
print("\n=== Test 2: Destination only in backup system ===")
response = await agent.run(
    "What flights are available to Berlin?",
    )
print(response)

## Selbstbewertungsmuster

Eine weitere Facette der Metakognition ist die **Selbstbewertung**: Ein separater Agent (oder derselbe Agent in einem zweiten Durchgang) überprüft eine Antwort auf Vollständigkeit, Genauigkeit und Nützlichkeit.

Im Folgenden erstellen wir einen `ResponseEvaluator`-Agenten, der Reisebüro-Antworten in drei Dimensionen bewertet.


In [ ]:
evaluation_agent = client.as_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="ResponseEvaluator",
    instructions="""You are a quality evaluator for travel agent responses.
Given a travel question and the agent's response, evaluate:
1. Completeness: Did it answer all parts of the question? (1-5)
2. Accuracy: Is the information correct? (1-5)
3. Helpfulness: Would a traveler find this useful? (1-5)
Provide a brief evaluation with scores and one suggestion for improvement.""",
)

# Evaluate the agent's response from Test 1
eval_prompt = f"""Question: What flights are available to Paris?
Agent Response: {response}

Please evaluate the above response."""

evaluation = await evaluation_agent.run(eval_prompt)
print("=== Self-Evaluation ===")
print(evaluation)

## Zusammenfassung

In dieser Lektion haben Sie gelernt, wie man **metakognitive Agenten** mit dem Microsoft Agent Framework erstellt:

- **Selbstreflexion**: Agenten, die ihr eigenes Denken überwachen und transparent kommunizieren, was passiert ist.
- **Fehlerbehebung mit Fallbacks**: Ein Muster mit Haupt- und Ersatzwerkzeug, bei dem der Agent Fehler (z.B. 404-Fehler) erkennt und automatisch eine alternative Quelle ausprobiert.
- **Selbstevaluation**: Ein separater Bewertungsagent, der Antworten auf Vollständigkeit, Genauigkeit und Hilfsbereitschaft bewertet.

Diese Muster machen Agenten robuster, transparenter und vertrauenswürdiger – entscheidende Eigenschaften für produktive Einsätze.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Haftungsausschluss**:
Dieses Dokument wurde mit dem KI-Übersetzungsdienst [Co-op Translator](https://github.com/Azure/co-op-translator) übersetzt. Obwohl wir uns um Genauigkeit bemühen, beachten Sie bitte, dass automatisierte Übersetzungen Fehler oder Ungenauigkeiten enthalten können. Das Originaldokument in seiner Ursprungssprache gilt als maßgebliche Quelle. Bei kritischen Informationen wird eine professionelle menschliche Übersetzung empfohlen. Wir übernehmen keine Haftung für Missverständnisse oder Fehlinterpretationen, die aus der Verwendung dieser Übersetzung entstehen.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
